# 03. Cribado virtual PAMPA

Este notebook muestra cómo usar el modelo final para predecir nuevas moléculas cuando ya tienes calculados los 11 descriptores alvaDesc/WEKA.

## Qué debe cambiar el usuario

Si tu profesor entrega 5 moléculas nuevas, primero debes calcular sus 11 descriptores con el mismo protocolo de alvaDesc/WEKA. Luego guarda un CSV y cambia `INPUT_FILE` en la celda **Configuración editable**.

El CSV debe contener estas columnas:

`LOGPcons`, `MACCSFP125`, `PCR`, `Psi_e_A`, `P_VSA_ppp_D`, `Mp`, `SpMin1_Bh(p)`, `SHED_AL`, `SM12_AEA(ri)`, `P_VSA_s_3`, `MATS5m`.

Opcionalmente puede tener una columna identificadora, por ejemplo `example_id`, y una columna `Actividad` si ya conoces la etiqueta experimental.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
%cd $ROOT

## Configuración editable

Para usar otra base, cambia `INPUT_FILE`. El resto puede quedar igual.

In [ ]:
INPUT_FILE = Path('data/query/example_positive_negative_11.csv')
OUTPUT_FILE = Path('results/screening/notebook_03_predictions.csv')
REPORT_JSON = Path('results/screening/notebook_03_report.json')
REPORT_MD = Path('results/screening/notebook_03_report.md')
ID_COLUMN = 'example_id'

## Crear ejemplo si no existe

Esta celda solo crea un ejemplo mínimo con una molécula positiva y una negativa del conjunto externo. Si ya tienes tu archivo, esta celda no lo modifica.

In [ ]:
if not INPUT_FILE.exists():
    external = pd.read_csv('data/raw/external_11.csv')
    example = pd.concat([
        external[external['Actividad'].eq('Act1')].head(1),
        external[external['Actividad'].eq('Act-1')].head(1),
    ], ignore_index=True)
    example.insert(0, ID_COLUMN, ['positive_external_example', 'negative_external_example'])
    INPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    example.to_csv(INPUT_FILE, index=False)

pd.read_csv(INPUT_FILE)

## Validar columnas

Antes de predecir, se revisa que estén los 11 descriptores requeridos.

In [ ]:
!python -m src.science_skills.pampa_computational_discovery validate-panel --input $INPUT_FILE

## Predecir permeabilidad

Este comando usa el modelo oficial `models/best_rf_pampa.pkl` y genera un CSV con probabilidades y clases predichas.

In [ ]:
!python -m src.science_skills.pampa_computational_discovery predict-panel --input $INPUT_FILE --output $OUTPUT_FILE --report-json $REPORT_JSON --report-md $REPORT_MD --id-column $ID_COLUMN

## Resultados

In [ ]:
pd.read_csv(OUTPUT_FILE)

## Siguiente paso

Para un análisis más completo con agentes locales sin API, continúa con el notebook `07_Prediccion_nuevas_moleculas_agentes_locales.ipynb`.